# Brreg Bedriftssøk (brønnøysundregistret)
Dette notebooken søker etter bedrifter i Enhetsregisteret basert på næringskode og kommune.

Kjør cellene **én etter én** fra topp til bunn.

## Steg 1– Installer og importer biblioteker
Nødvendige biblioteker:
- `requests` (for å sende API-kall)
- `pandas` (for å lage tabeller)

In [19]:
import requests
import pandas as pd

print('Biblioteker importert!')

Biblioteker importert!


## Steg 2 – Definer URL og innstillinger
Setter opp basis-URL-en til Brreg sitt API og velger næringskode og kommune du vil søke på.

In [20]:
BASE_URL = 'https://data.brreg.no/enhetsregisteret/api/enheter'

# Søke kriterier
NAERINGSKODE   = '46.900'      # Næringskode
KOMMUNE        = 'Trondheim'   # Kommune navn
MAKS_SVAR      = 20            # Antall resultater

print(f'Klar til å søke etter næringskode {NAERINGSKODE} i {KOMMUNE}')

Klar til å søke etter næringskode 46.900 i Trondheim


## Steg 3 – Hent kommunenummer
Brreg sitt API bruker kommunenummer (4 siffer) i stedet for kommunenavn som tekst.
Vi må derfor hente ut en liste, gjennom Brreg API, over alle kommuner og finne nummeret automatisk.

In [21]:
def hent_kommunenummer(kommunenavn: str) -> str | None:
    side = 0
    while True:
        url = f'https://data.brreg.no/enhetsregisteret/api/kommuner?page={side}&size=100'
        r = requests.get(url, headers={'Accept': 'application/json'})
        
        if r.status_code != 200:
            print(f'Kunne ikke hente kommuneliste: {r.status_code}')
            return None
        
        data = r.json()
        kommuner = data.get('_embedded', {}).get('kommuner', [])
        
        for k in kommuner:
            if k.get('navn', '').upper() == kommunenavn.upper(): 
                return k.get('nummer')                           
        
        total_sider = data.get('page', {}).get('totalPages', 1)
        side += 1
        if side >= total_sider:
            break
    
    return None

# Test 
kommunenr = hent_kommunenummer(KOMMUNE)

if kommunenr:
    print(f'Kommunenummer for {KOMMUNE}: {kommunenr}')
else:
    print(f'Fant ikke kommunenummer for "{KOMMUNE}". Sjekk stavemåten.')

Kommunenummer for Trondheim: 5001


## Steg 4 – Send API-forespørsel til Brreg
Sender forespørsel om innhentning av data med næringskode og kommunenummer som filtre.
Skriver ut rå-JSON-svaret så du kan se hva API-et faktisk returnerer.

In [22]:
params = {
    'naeringskode':  NAERINGSKODE,
    'kommunenummer': kommunenr,
    'size':          MAKS_SVAR,
    'page':          0
}

response = requests.get(BASE_URL, params=params, headers={'Accept': 'application/json'})

print(f'Status: {response.status_code}')

if response.status_code == 200:
    data = response.json()
    enheter = data.get('_embedded', {}).get('enheter', [])
    totalt  = data.get('page', {}).get('totalElements', 0)
    print(f'Fant totalt {totalt} bedrifter i registeret. Hentet {len(enheter)} nå.')
else:
    print(f'Feil: {response.text}')

Status: 200
Fant totalt 27 bedrifter i registeret. Hentet 20 nå.


## Steg 5 – Data i JSON fil 
Få forståelse av data. JSON fil er ikke så vanskelig å forstå men det er viktig å se hvilken informasjon som er tilgjengelig 

In [12]:
if enheter:
    print('Rå data for første bedrift:')
    import json
    print(json.dumps(enheter[0], indent=2, ensure_ascii=False))
else:
    print('Ingen bedrifter å vise.')

Rå data for første bedrift:
{
  "organisasjonsnummer": "915637450",
  "navn": "4POTER CATHRINE UDE",
  "organisasjonsform": {
    "kode": "ENK",
    "beskrivelse": "Enkeltpersonforetak",
    "_links": {
      "self": {
        "href": "https://data.brreg.no/enhetsregisteret/api/organisasjonsformer/ENK"
      }
    }
  },
  "registreringsdatoEnhetsregisteret": "2015-07-08",
  "registrertIMvaregisteret": false,
  "naeringskode1": {
    "kode": "46.900",
    "beskrivelse": "Uspesifisert engroshandel"
  },
  "harRegistrertAntallAnsatte": false,
  "epostadresse": "hundepratuljen.as@gmail.com",
  "mobil": "905 20 698",
  "forretningsadresse": {
    "land": "Norge",
    "landkode": "NO",
    "postnummer": "7046",
    "poststed": "TRONDHEIM",
    "adresse": [
      "Gina Krogs veg 24"
    ],
    "kommune": "TRONDHEIM",
    "kommunenummer": "5001"
  },
  "institusjonellSektorkode": {
    "kode": "8200",
    "beskrivelse": "Personlig næringsdrivende"
  },
  "registrertIForetaksregisteret": false

## Celle 6 – Pakk ut og strukturer dataene
Analyere og optimere dataen 

In [23]:
resultater = []

for enhet in enheter:
    adresse_obj  = enhet.get('forretningsadresse') or enhet.get('postadresse') or {}
    adresse_str  = ', '.join([a for a in adresse_obj.get('adresse', []) if a])
    poststed     = adresse_obj.get('poststed', '')
    postnr       = adresse_obj.get('postnummer', '')
    full_adresse = f'{adresse_str}, {postnr} {poststed}'.strip(', ')

    resultater.append({
        'Navn':        enhet.get('navn', '-'),
        'Org.nr':      enhet.get('organisasjonsnummer', '-'),
        'Næringskode': enhet.get('naeringskode1', {}).get('kode', '-'),
        'Beskrivelse': enhet.get('naeringskode1', {}).get('beskrivelse', '-'),
        'Adresse':     full_adresse or '-',
        'Telefon':     enhet.get('telefon') or enhet.get('mobil') or '-',
        'E-post':      enhet.get('epostadresse') or '-',
        'Hjemmeside':  enhet.get('hjemmeside') or '-',
        'Ansatte':     enhet.get('antallAnsatte', '-'),
    })

print(f'Pakket ut {len(resultater)} bedrifter.')

Pakket ut 20 bedrifter.


## Celle 7 – Vis som tabell med Pandas
Her lager vi en DataFrame (tabell) av dataene og viser dem pent i notebooken.

In [26]:
df = pd.DataFrame(resultater)

# Vis alle kolonner uten avkorting
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)

df.head()

,Navn,Org.nr,Næringskode,Beskrivelse,Adresse,Telefon,E-post,Hjemmeside,Ansatte
0,4POTER CATHRINE UDE,915637450,46.900,Uspesifisert engroshandel,"Gina Krogs veg 24, 7046 TRONDHEIM",905 20 698,hundepratuljen.as@gmail.com,-,-
1,AKTIV SIKKERHETSSYSTEMER ANS,964540942,46.900,Uspesifisert engroshandel,"Kjøpmannsgata 9, 7013 TRONDHEIM",-,-,-,-
2,AMPA AS,929480759,46.900,Uspesifisert engroshandel,"Peder Falcks veg 8, 7044 TRONDHEIM",-,-,-,-
3,ARACO AS,984290969,46.900,Uspesifisert engroshandel,"Kalvskinnsgata 3, 7012 TRONDHEIM",93245168,roe.strommen@araco.no,-,-
4,AURORA TRADING AS,996933822,46.900,Uspesifisert engroshandel,"Eventyrvegen 53, 7056 RANHEIM",930 17 083,-,-,-


## Celle 8 – Lagre til CSV
Her lagrer vi resultattabellen til en CSV-fil du kan åpne i Excel.

In [27]:
filnavn = f"bedrifter_{NAERINGSKODE.replace('.','_')}_{KOMMUNE}.csv"
df.to_csv(filnavn, index=False, encoding='utf-8-sig')
print(f'✅ Lagret til {filnavn}')

✅ Lagret til bedrifter_46_900_Trondheim.csv
